# Model Based System Engineering Assistant: 
## Leveraging AI for System Engineering models 

Welcome to the **MBSE Assistant Notebook**, a powerful tool designed to augment your ability to review, comprehend, and enhance system models with AI-driven support. This notebook integrates a variety of capabilities to assist with:

1. **Model Review and Comprehension**:
   - Gain deeper insights into the structure and behavior of your system model.
   - Use AI to clarify relationships, dependencies, and interactions within the model.

2. **Impact Analysis**:
   - Explore the potential effects of component failures on the system.
   - Simulate scenarios to identify vulnerabilities and design improvements.

3. **Simulation Generation**:
   - Automatically generate simulations based on the model's architecture and behaviors.
   - Use simulations for validation, testing, and demonstration purposes.

4. **Failure Analysis**:
   - Assess the system's resilience under various failure conditions.
   - Extract meaningful insights to strengthen reliability and performance.

5. **Content Extraction for Downstream Domains**:
   - Convert model data into formats suitable for use in other engineering domains.
   - Facilitate seamless collaboration across teams and tools.

This notebook serves as a comprehensive platform to bridge the gap between system models and actionable insights, empowering architects to make informed decisions and design robust solutions. Whether you're investigating enhancements, running failure simulations, or preparing data for external domains, this assistant streamlines the process with AI-enhanced guidance.

Feel free to explore and adapt the prompts and models to suit your specific use case. Let’s dive in and unlock the potential of your system model!


### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "Mountain_Bike": "Mountain_Bike/Mountain Bike",
    "Bike BrakeSystem Library": {"path" : "/home/simcenter/studio/Bike_BrakeSystem_Library/Bike BrakeSystem Library"}
}
path_to_model = "../Mountain Bike.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../Mountain Bike.traceability")


## 🔄 Embedding Generation Process

### Overview
This section generates embeddings, which streamlines processing and analyzing the model. The embeddings provide a structured representation of the data that powers subsequent tasks and visualizations.

### When is this step necessary?
- **Initial Run:** If this notebook is being run for the first time, embeddings will need to be generated.
- **Model Updates:** If the underlying model has been modified, regenerating embeddings ensures the data remains accurate and up-to-date.

### Important Notes
- **Duration:** This process may take **a minute or more** to complete, depending on the size and complexity of the model.
- **Output:** Upon completion, embeddings will be ready for use in subsequent sections of the notebook.

### Instructions
1. Ensure the required model is loaded.
2. Run the cell below to initiate the embedding generation process.
3. Wait for the process to complete before proceeding.

> 💡 **Tip:** You can monitor progress in the notebook's output cell. If any errors occur, check that the model is properly configured and accessible.

---

> ⚠️ **Warning:** Interrupting this step may result in incomplete or invalid embeddings. If interrupted, re-run the cell to restart the process.

---

Continue to the next cell to generate embeddings. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Loading embeddings
embeddings loaded.


## 🎯 Prompt for System Model Element Analysis


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[ocb] Mountain Bike", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [OCB] Mountain Bike, Similarity: 0.70, Type: Diagram, Phase: Operational Analysis OA, Source: , Target: 






## 📝 Generate Structured Input File 

### Purpose
This cell generates a structured structured input file, `capella_model.yaml`, which serves as input for prompts tailored to the object type of the last selected item. This ensures that the prompt aligns with the model's structure and provides accurate context for analysis.

### What This Cell Does
- Extracts data related to the elements selected.
- Structures the data into a YAML format.
- Saves the YAML file as `capella_model.yaml` for further use.

### Key Features
- The YAML file includes:
  - **Object Type:** The type of the element.
  - **Object Details:** Attributes, relationships, and associated exchanges.
  - **Structured Input:** Designed to enhance prompt accuracy and model analysis.

### Instructions
1. Run this cell to create the `capella_model.yaml` file.
2. Ensure the last selected item is correctly identified to avoid mismatched data.
3. Use the generated YAML file as structured input for tailored prompts in subsequent steps.

### Notes
- The file will be saved in the current working directory.
- If the selected object changes, re-run this cell to update the YAML file.

> 💡 **Tip:** Review the contents of `capella_model.yaml` if needed to ensure accuracy and completeness.


In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects : 

    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  



yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## 🖼️ Generate Contextual Diagram

### Purpose
This cell generates a **contextual diagram** for the selected object. The diagram visually represents the object's relationships and interactions, providing a clear and comprehensive view of its context within the model.

### Supported Object Types
This functionality works the following ARCADIA elements, including:
- **Functions**
- **Activities**
- **Components**
- **Entities**
- **Exchanges and Interactions**
- **Physical Links**



In [5]:
for object in  selected_objects : 
    #print(object)
    if object["type"] == "Diagram" :
        diagram = model.by_uuid(object["uuid"])
        #display(diagram)
    else:
        obj =  model.by_uuid(object["uuid"])
        capellambse_helper.display_context_diagram(obj)


## ⚙️ Execute Followup Prompts
This cell creates an AI agent that going to be used to perfrom the analysis and loads the stuctured content. 

In [6]:

from capella_tools import Open_AI_RAG_manager   
yaml_content = yaml_handler.get_yaml_content()
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)

OpenAI API Key retrieved successfully.


## ⚙️ Execute Followup Prompts

### Purpose
This cell executes the **follow up prompts** defined for analyzing the model. These prompts can be added by the note book user in addition to the default prompts

### What This Cell Does
- Use the structured input file (e.g., `capella_model.yaml`).
- Excutes the prompt defined in the cell.
- Outputs the results directly in the notebook.

### 🛠️ Instructions
1. Ensure the `capella_model.yaml` file is generated and up-to-date.
2. Run this cell to execute the default prompts.
3. Review the output to verify the accuracy and relevance of the analysis.

> 💡 **Tip:** Modify the promt to your needs.

In [12]:
prompt = """
Review each operational capability, verify that they have an appropriate operational process.
State the Operational Capability
List the associated processes and a summary.
Give a table with column on the process name, and assessment on whether it alings with capability.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()



**Your prompt:** 
Review each operational capability, verify that they have an appropriate operational process.
State the Operational Capability
List the associated processes and a summary.
Give a table with column on the process name, and assessment on whether it alings with capability.


**ChatGPT Response:**

Here's a detailed review of each operational capability, its associated operational processes, and an assessment of alignment:

### 1. Terrain Navigation
**Associated Processes:**
- Improve Terrain Navigation
- Ride Mountain Trails
- Maintain Bike
- Terrain Adjust

**Summary:**
Terrain Navigation involves guiding the bike efficiently over various terrain. All associated processes support this, focusing on enhancing navigation, maintaining equipment for performance efficiency, and making necessary terrain adjustments to improve the ride experience.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Improve Terrain Navigation  | Aligns - Focuses on enhancing navigation capabilities. |
| Ride Mountain Trails        | Aligns - Core to navigation on terrain.     |
| Maintain Bike               | Aligns - Ensures bike readiness for navigation.  |
| Terrain Adjust              | Aligns - Includes adjusting for improved terrain handling. |

### 2. Durability and Shock Absorption
**Associated Processes:**
- Ride Mountain Trails
- Maintain Bike
- Enhance Frame and Suspension
- Terrain Adjust

**Summary:**
This capability focuses on the bike's ability to withstand stress and absorb shocks. Processes like maintaining equipment and enhancing physical components directly support this, ensuring the bike can endure rough conditions and provide a comfortable ride.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Ride Mountain Trails        | Aligns - Tests and validates durability.   |
| Maintain Bike               | Aligns - Ensures ongoing durability through maintenance. |
| Enhance Frame and Suspension| Aligns - Directly enhances shock absorption. |
| Terrain Adjust              | Aligns - Includes handling stress from terrain. |

### 3. Climatic Adaptability
**Associated Processes:**
- Ride Mountain Trails
- Maintain Bike
- Terrain Adjust
- Improve for Weather

**Summary:**
Climatic Adaptability ensures the bike can perform well in diverse weather conditions. Processes involving weather improvements and routine maintenance directly contribute to this capability by adapting the bike components for various climates.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Ride Mountain Trails        | Aligns - Validates capability across climates. |
| Maintain Bike               | Aligns - Ensures continuity in performance. |
| Terrain Adjust              | Partially Aligns - Involves adjustments for adaptability. |
| Improve for Weather         | Aligns - Directly improves adaptability to weather. |

### 4. Maneuverability
**Associated Processes:**
- Ride Mountain Trails
- Maintain Bike
- Terrain Adjust
- Optimize for Balance

**Summary:**
Maneuverability focuses on the bike's control and ease of handling over diverse terrains. Processes that maintain bike condition and optimize balance contribute directly to improving rider control and maneuverability.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Ride Mountain Trails        | Aligns - Demonstrates maneuverability effectiveness. |
| Maintain Bike               | Aligns - Keeps components in prime working order. |
| Terrain Adjust              | Aligns - Enhances bike control on terrain.  |
| Optimize for Balance        | Aligns - Improves overall control and ease of movement. |

### 5. Mountain Trail Ride
**Associated Processes:**
- Ride Mountain Trails

**Summary:**
This capability is directly related to the process of riding over mountain trails. It ensures the bike's performance matches the demands of such challenging environments.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Ride Mountain Trails        | Aligns - Direct core element validating the capability. |

### 6. Terrain Adjustment
**Associated Processes:**
- Terrain Adjust

**Summary:**
Terrain Adjustment involves adapting the bike to handle differing terrain types. The singular focus process supports making precise adjustments that are required for optimal performance across various trail conditions.

| Process Name                | Alignment Assessment                        |
|-----------------------------|---------------------------------------------|
| Terrain Adjust              | Aligns - Solely dedicated to this capability. |

**Conclusion:** Each operational capability has the appropriate processes assigned, ensuring efficiency in meeting each capability's objectives and supporting the Mountain Bike's performance and robustness across various conditions and terrains.

**Token Usage Info:**

Tokens used: prompt=52907, completion=845, total=53752

In [9]:
prompt = """
Please create a set of 10 functional requirements for the Mountain Bike related functional chains. 
Format the requirement in the following way:
1. Each requirement will have a heading and body.
2. The heading should be limited to 5 words.
3. The body should be written as a shall statement.
4. The body to be clear, concise and testable.
5. Reference the functional chain.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()


**Your prompt:** 
Please create a set of 10 functional requirements for the Mountain Bike related functional chains. 
Format the requirement in the following way:
1. Each requirement will have a heading and body.
2. The heading should be limited to 5 words.
3. The body should be written as a shall statement.
4. The body to be clear, concise and testable.
5. Reference the functional chain.


**ChatGPT Response:**

Here is a set of 10 functional requirements for the Mountain Bike related to functional chains, formatted as specified:

1. **Gear Selection Display**
   - **Body**: The Mountain Bike shall display the current gear during operation. *(Related to Functional Chain: Gear Selection)*

2. **Power Management Control**
   - **Body**: The Mountain Bike shall automatically regulate power output during varied terrain ascents. *(Related to Functional Chain: Terrain Navigation)*

3. **Brake System Integration**
   - **Body**: The Mountain Bike shall engage both front and rear brakes within 0.5 seconds of control input. *(Related to Functional Chain: Brake Efficiency)*

4. **Suspension Adjustment Mechanism**
   - **Body**: The Mountain Bike suspension shall adjust to provide optimal damping based on trail roughness. *(Related to Functional Chain: Vibration Dampening)*

5. **Lighting Activation Control**
   - **Body**: The Mountain Bike shall automatically activate lighting under low-light conditions. *(Related to Functional Chain: Visibility in Low Light)*

6. **Battery Status Display**
   - **Body**: The Mountain Bike shall display battery levels for all powered accessories. *(Related to Functional Chain: Battery Life Management)*

7. **Climbing Assistance Feature**
   - **Body**: The Mountain Bike shall provide additional torque support on inclines greater than 15%. *(Related to Functional Chain: Climb Support)*

8. **Weather Adaptation System**
   - **Body**: The Mountain Bike shall adapt component functions for operation in wet conditions. *(Related to Functional Chain: Weather Adaptability)*

9. **Load-Bearing Test Capability**
   - **Body**: The Mountain Bike shall conduct real-time load assessments during rides. *(Related to Functional Chain: Load Management)*

10. **Noise Reduction Mechanism**
    - **Body**: The Mountain Bike shall minimize operational noise through vibration isolators. *(Related to Functional Chain: Noise Reduction)*

**Token Usage Info:**

Tokens used: prompt=52448, completion=403, total=52851

## 💬 Launch Interactive Chat on Structured Input

### Purpose
This cell launches an interactive chat session based on the structured input file, leveraging ARCADIA and Polarion terminology to ensure the analysis remains consistent with the modeling context.

### Key Features
- Uses the generated structured input file (e.g., `capella_model.yaml`) to guide the conversation.
- Supports ARCADIA terms such as **functions**, **components**, **activities**, and **exchanges**.
- Incorporates Polarion terms like **workitem**, **requirement**, and **traceability** for seamless integration with requirements management workflows.

### How It Works
1. The structured input file provides context for the chat session.
2. You can interactively query details about the model elements, relationships, and dependencies.
3. Responses are tailored using ARCADIA and Polarion terminology.

### Example Prompts
- **For ARCADIA:**
  ```plaintext
  What are the dependencies and exchanges for the selected function in the Logical Architecture phase?

- **For Polarion:**
  ```plaintext
  Provide the traceability matrix for the workitem linked to this component.
  ```
### 🛠️ Instructions
1. Ensure the `capella_model.yaml` file has been generated and is up-to-date.
2. Run this cell to start the interactive chat session.
3. Enter your queries using **ARCADIA** or **Polarion** terms for accurate and relevant responses.

### 📝 Notes
- The interactive chat dynamically adapts to the content in the structured input file.
- Use precise and contextual queries to ensure the best results and insights.

> 💡 **Tip:** If you're unfamiliar with ARCADIA or Polarion terminology, consult the respective documentation or glossary for guidance.



In [10]:
analyzer.interactive_chat()


Starting interactive chat...


Output()

Textarea(value='', layout=Layout(border_bottom='2px solid #4A90E2', border_left='2px solid #4A90E2', border_ri…

Dropdown(description='Load file:', layout=Layout(width='auto'), options=('', 'embeddings.json', 'capella_model…

Waiting for chat interactions...


In [11]:
print("Done")

Done


# 